In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [4]:
len(documents)

72

In [5]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [6]:
question = "How does the agentic loop keep calling the model until it stops?"

In [10]:
def search(question):

    return index.search(
        question,
        num_results=1
    )

In [11]:
search_results = search(question)

In [15]:
search_results[0]['filename']

'01-agentic-rag/lessons/14-agentic-loop.md'

In [16]:
from dotenv import load_dotenv
load_dotenv()

from hw_rag_helper import RAGBase
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://api.deepseek.com",
    timeout=300
)

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [18]:
answer = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

('The agentic loop keeps calling the model by running a `while True` loop. Inside the loop, it sends the current message history (including tool results) to the model. After each response, it checks whether the model returned any `function_call` items. If there are function calls, the loop runs the corresponding tool (like `search`), appends the tool result to the message history, and then continues to the next iteration—calling the model again with the updated history. The loop stops when the model returns a response that contains **no** function calls, meaning the model has produced a final answer and does not need any more tool executions. This is controlled by a `has_function_calls` flag that is set to `True` if any function call is found; the loop breaks when this flag is `False` after processing a response.', 7298)


In [19]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

In [23]:
index_ch = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index_ch.fit(chunks)

In [24]:
assistant_ch = RAGBase(
    index=index_ch,
    llm_client=openai_client,
)

In [25]:
answer_ch = assistant_ch.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

('The agentic loop keeps calling the model by running a `while True` loop. Inside the loop, it sends the current message history (including tool results) to the model. After each response, it checks whether the model returned any `function_call` items. If there are function calls, the loop runs the corresponding tool (like `search`), appends the tool result to the message history, and then continues to the next iteration—calling the model again with the updated history. The loop stops when the model returns a response that contains **no** function calls, meaning the model has produced a final answer and does not need any more tool executions. This is controlled by a `has_function_calls` flag that is set to `True` if any function call is found; the loop breaks when this flag is `False` after processing a response.', 7298)


In [26]:
from toyaikit.tools import Tools
from toyaikit.llm import OpenAIChatCompletionsClient
from toyaikit.chat.runners import OpenAIChatCompletionsRunner, DisplayingRunnerCallback
from toyaikit.chat import IPythonChatInterface

In [27]:
instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
""".strip()

In [28]:
def search(query: str) -> dict[str, str]:
    """
    Search the answer from lesson content.
    """
    return index.search(
        query,
        num_results=5
    )

In [29]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [30]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

# Define the model to use
llm_client = OpenAIChatCompletionsClient(
    model='deepseek-v4-flash',
    client=openai_client
)

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIChatCompletionsRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client
)

In [31]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


Aspect,Plain RAG,Agentic Loop
Who decides the flow?,The developer — steps are fixed in code,The LLM — it decides what to do at each step
Number of searches,"Exactly one, with the raw user query",As many as the model decides it needs
Recovery from errors,"❌ None — if search returns garbage, the LLM gets garbage","✅ Can fix typos, rephrase queries, search again"
"Example: ""How do I run Olama?""","Searches for ""Olama"" — finds nothing — says ""I don't know""","Searches ""Olama"" → sees bad results → re-searches ""Ollama"" → finds the answer"
Cost,One API call per question,Multiple API calls (each round-trip costs tokens)
Latency,Lower — one round-trip,Higher — multiple round-trips
Predictability,High — always the same path,Lower — the model may take different paths on different runs
